In [3]:
"""
Maize Yield Forecasting in Nigeria
Version B: State-aware + Tuned + Lagged Target Features
"""

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import warnings
import os
import random

warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ==========================================================
# Config
# ==========================================================
class Config:
    SEED = 42
    SEQ_LENGTH = 5
    PRED_LENGTH = 1
    BATCH_SIZE = 32
    EPOCHS = 100
    PATIENCE = 15
    N_TRIALS_DL = 5
    N_TRIALS_RF = 8

    FEATURE_COLS = ['LST', 'NDVI', 'Precipitation', 'Temperature',
                    'year_sin', 'year_cos']
    LAG_COLS = ['value_lag1', 'value_lag2']
    ALL_FEATURES = FEATURE_COLS + LAG_COLS

    TARGET_COL = 'value'
    STATE_COL = 'admin_1'

    VAL_YEARS = 2
    TEST_YEARS = 3

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(Config.SEED)
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)


# ==========================================================
# Data preparation
# ==========================================================
def add_lag_features(df, target_col, state_col, lags=(1, 2)):
    df = df.sort_values([state_col, 'year']).reset_index(drop=True)
    for lag in lags:
        df[f'{target_col}_lag{lag}'] = df.groupby(state_col)[target_col].shift(lag)
    df = df.dropna(subset=[f'{target_col}_lag{l}' for l in lags]).reset_index(drop=True)
    return df


def create_sequences(df, feature_cols, target_col, state_col, seq_len=5, pred_len=1):
    X_list, y_list, state_list = [], [], []
    for state in df[state_col].unique():
        sdf = df[df[state_col] == state].sort_values('year').reset_index(drop=True)
        if len(sdf) < seq_len + pred_len:
            continue
        features = sdf[feature_cols + [target_col]].values
        for i in range(len(sdf) - seq_len - pred_len + 1):
            X_list.append(features[i:i + seq_len, :-1])
            y_list.append(features[i + seq_len, -1])
            state_list.append(state)
    return np.array(X_list), np.array(y_list), np.array(state_list)


def walk_forward_split_3way(states, val_years=2, test_years=3):
    train_idx, val_idx, test_idx = [], [], []
    for state in np.unique(states):
        idx = np.where(states == state)[0]
        if len(idx) <= val_years + test_years:
            continue
        test_idx.extend(idx[-test_years:])
        val_idx.extend(idx[-(test_years + val_years):-test_years])
        train_idx.extend(idx[:-(test_years + val_years)])
    return np.array(train_idx), np.array(val_idx), np.array(test_idx)


# ==========================================================
# Models
# ==========================================================
class LSTMModel(nn.Module):
    def __init__(self, input_size, num_states, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size + num_states, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, x, state_oh):
        out, _ = self.lstm(x)
        combined = torch.cat([out[:, -1, :], state_oh], dim=1)
        return self.fc(combined).squeeze(-1)


class GRUModel(nn.Module):
    def __init__(self, input_size, num_states, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers,
                          batch_first=True,dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size + num_states, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, x, state_oh):
        out, _ = self.gru(x)
        combined = torch.cat([out[:, -1, :], state_oh], dim=1)
        return self.fc(combined).squeeze(-1)


class BiLSTMAttention(nn.Module):
    def __init__(self, input_size, num_states, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.bilstm = nn.LSTM(input_size, hidden_size, num_layers,
                              batch_first=True,
                              dropout=dropout if num_layers > 1 else 0.0,
                              bidirectional=True)
        self.attention = nn.Linear(hidden_size * 2, 1)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2 + num_states, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, x, state_oh):
        out, _ = self.bilstm(x)
        attn = torch.softmax(self.attention(out), dim=1)
        context = (out * attn).sum(dim=1)
        combined = torch.cat([context, state_oh], dim=1)
        return self.fc(combined).squeeze(-1)


class TCNModel(nn.Module):
    def __init__(self, input_size, num_states, num_channels=(32, 64, 64),
                 kernel_size=3, dropout=0.2):
        super().__init__()
        layers = []
        in_ch = input_size
        for out_ch in num_channels:
            layers.extend([
                nn.Conv1d(in_ch, out_ch, kernel_size, padding=kernel_size - 1),
                nn.BatchNorm1d(out_ch), nn.ReLU(), nn.Dropout(dropout)])
            in_ch = out_ch
        self.tcn = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(in_ch + num_states, 1)

    def forward(self, x, state_oh):
        x = x.transpose(1, 2)
        x = self.tcn(x)
        x = self.pool(x).squeeze(-1)
        combined = torch.cat([x, state_oh], dim=1)
        return self.fc(combined).squeeze(-1)


def build_model(name, input_size, num_states, params):
    hs = params['hidden_size']
    nl = params['num_layers']
    dr = params['dropout']
    if name == 'LSTM':
        return LSTMModel(input_size, num_states, hs, nl, dr)
    if name == 'GRU':
        return GRUModel(input_size, num_states, hs, nl, dr)
    if name == 'Bi-LSTM + Attn':
        return BiLSTMAttention(input_size, num_states, hs, nl, dr)
    if name == 'TCN':
        ch = (max(hs // 2, 8), hs, hs)
        return TCNModel(input_size, num_states, num_channels=ch, dropout=dr)
    raise ValueError(name)


# ==========================================================
# Dataset & training
# ==========================================================
class MaizeDataset(Dataset):
    def __init__(self, X, y, state_oh):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.state_oh = torch.FloatTensor(state_oh)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.state_oh[idx], self.y[idx]


def train_model(model, train_loader, val_loader, epochs, patience, lr, device):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)
    criterion = nn.MSELoss()

    best_val, best_state, patience_counter = float('inf'), None, 0
    for _ in range(epochs):
        model.train()
        for xb, sb, yb in train_loader:
            xb = xb.to(device)
            sb = sb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb, sb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, sb, yb in val_loader:
                xb = xb.to(device)
                sb = sb.to(device)
                yb = yb.to(device)
                val_loss += criterion(model(xb, sb), yb).item()
        val_loss /= max(len(val_loader), 1)
        scheduler.step(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val


def predict(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, sb, _ in loader:
            preds.append(model(xb.to(device), sb.to(device)).cpu().numpy())
    return np.concatenate(preds) if preds else np.array([])


# ==========================================================
# Main
# ==========================================================
def main():
    print("=" * 70)
    print("🌽 Maize Yield Forecasting — Version B (with lag features)")
    print("=" * 70)
    print(f"🔥 Device: {Config.DEVICE}")

    df = pd.read_csv("maize_yearly_analysis.csv")
    df = df.sort_values([Config.STATE_COL, 'year']).reset_index(drop=True)
    df = add_lag_features(df, Config.TARGET_COL, Config.STATE_COL, lags=(1, 2))
    print(f"\n📊 Data after lags: {df.shape} | States: {df[Config.STATE_COL].nunique()}")

    X, y, states = create_sequences(
        df, Config.ALL_FEATURES, Config.TARGET_COL, Config.STATE_COL,
        Config.SEQ_LENGTH, Config.PRED_LENGTH)
    print(f"🔧 Sequences: X={X.shape}, y={y.shape}")
    print(f"🧾 Features used: {Config.ALL_FEATURES}")

    train_idx, val_idx, test_idx = walk_forward_split_3way(
        states, Config.VAL_YEARS, Config.TEST_YEARS)
    print(f"✂️  Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

    X_train, y_train, s_train = X[train_idx], y[train_idx], states[train_idx]
    X_val, y_val, s_val = X[val_idx], y[val_idx], states[val_idx]
    X_test, y_test, s_test = X[test_idx], y[test_idx], states[test_idx]

    try:
        encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    except TypeError:
        encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
    s_train_oh = encoder.fit_transform(s_train.reshape(-1, 1))
    s_val_oh = encoder.transform(s_val.reshape(-1, 1))
    s_test_oh = encoder.transform(s_test.reshape(-1, 1))
    num_states = s_train_oh.shape[1]
    print(f"🏷️  State one-hot dim: {num_states}")

    n_feat = len(Config.ALL_FEATURES)
    n_lag = len(Config.LAG_COLS)
    n_base = n_feat - n_lag

    scaler_X = StandardScaler()
    X_train_scaled = X_train.copy().astype(float)
    X_val_scaled = X_val.copy().astype(float)
    X_test_scaled = X_test.copy().astype(float)

    X_train_scaled[..., :n_base] = scaler_X.fit_transform(
        X_train[..., :n_base].reshape(-1, n_base)).reshape(X_train.shape[0], -1, n_base)
    X_val_scaled[..., :n_base] = scaler_X.transform(
        X_val[..., :n_base].reshape(-1, n_base)).reshape(X_val.shape[0], -1, n_base)
    X_test_scaled[..., :n_base] = scaler_X.transform(
        X_test[..., :n_base].reshape(-1, n_base)).reshape(X_test.shape[0], -1, n_base)

    y_train_scaled = y_train.astype(float)
    y_val_scaled = y_val.astype(float)

    input_size = X_train_scaled.shape[-1]

    DL_SPACE = {
        'hidden_size': [32, 64, 128],
        'num_layers': [1, 2],
        'dropout': [0.1, 0.2, 0.3],
        'lr': [1e-3, 5e-4],
    }
    dl_keys = list(DL_SPACE.keys())
    model_names = ['LSTM', 'GRU', 'Bi-LSTM + Attn', 'TCN']
    best_dl_params = {}

    for name in model_names:
        print(f"\n🔎 Tuning {name}...")
        best_val_loss = float('inf')
        best_params = None
        for trial in range(Config.N_TRIALS_DL):
            params = {k: random.choice(DL_SPACE[k]) for k in dl_keys}
            set_seed(Config.SEED + trial)
            model = build_model(name, input_size, num_states, params)
            train_loader = DataLoader(MaizeDataset(X_train_scaled, y_train_scaled, s_train_oh),
                batch_size=Config.BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(
                MaizeDataset(X_val_scaled, y_val_scaled, s_val_oh),
                batch_size=Config.BATCH_SIZE, shuffle=False)
            _, vl = train_model(model, train_loader, val_loader,
                                Config.EPOCHS, Config.PATIENCE,
                                params['lr'], Config.DEVICE)
            print(f"   trial {trial + 1}/{Config.N_TRIALS_DL}: "
                  f"val_loss={vl:.4f} | params={params}")
            if vl < best_val_loss:
                best_val_loss = vl
                best_params = params
        print(f"   ✅ Best {name}: val_loss={best_val_loss:.4f} | {best_params}")
        best_dl_params[name] = best_params

    print("\n🔁 Retraining best DL models on train + val...")
    X_trainval = np.concatenate([X_train_scaled, X_val_scaled], axis=0)
    y_trainval = np.concatenate([y_train_scaled, y_val_scaled], axis=0)
    s_trainval_oh = np.concatenate([s_train_oh, s_val_oh], axis=0)

    cut = int(len(X_trainval) * 0.85)
    tr_loader = DataLoader(
        MaizeDataset(X_trainval[:cut], y_trainval[:cut], s_trainval_oh[:cut]),
        batch_size=Config.BATCH_SIZE, shuffle=True)
    vs_loader = DataLoader(
        MaizeDataset(X_trainval[cut:], y_trainval[cut:], s_trainval_oh[cut:]),
        batch_size=Config.BATCH_SIZE, shuffle=False)

    final_dl_models = {}
    for name in model_names:
        set_seed(Config.SEED)
        model = build_model(name, input_size, num_states, best_dl_params[name])
        model, _ = train_model(model, tr_loader, vs_loader,
                               Config.EPOCHS, Config.PATIENCE,
                               best_dl_params[name]['lr'], Config.DEVICE)
        final_dl_models[name] = model

    print("\n🔎 Tuning Random Forest...")
    X_train_rf = np.concatenate(
        [X_train_scaled.reshape(len(X_train_scaled), -1), s_train_oh], axis=1)
    X_val_rf = np.concatenate(
        [X_val_scaled.reshape(len(X_val_scaled), -1), s_val_oh], axis=1)
    X_test_rf = np.concatenate(
        [X_test_scaled.reshape(len(X_test_scaled), -1), s_test_oh], axis=1)

    RF_SPACE = {
        'n_estimators': [100, 200, 400],
        'max_depth': [4, 6, 10, None],
        'min_samples_leaf': [1, 2, 4],
    }
    best_rf_mse, best_rf_params = float('inf'), None
    for trial in range(Config.N_TRIALS_RF):
        params = {k: random.choice(RF_SPACE[k]) for k in RF_SPACE}
        rf = RandomForestRegressor(random_state=Config.SEED, n_jobs=-1, **params)
        rf.fit(X_train_rf, y_train)
        val_mse = mean_squared_error(y_val, rf.predict(X_val_rf))
        print(f"   trial {trial + 1}/{Config.N_TRIALS_RF}: "
              f"val_mse={val_mse:.4f} | params={params}")
        if val_mse < best_rf_mse:
            best_rf_mse = val_mse
            best_rf_params = params
    print(f"   ✅ Best RF: val_mse={best_rf_mse:.4f} | {best_rf_params}")

    X_trainval_rf = np.concatenate([X_train_rf, X_val_rf], axis=0)
    y_trainval_raw = np.concatenate([y_train, y_val], axis=0)
    rf_final = RandomForestRegressor(random_state=Config.SEED, n_jobs=-1, **best_rf_params)
    rf_final.fit(X_trainval_rf, y_trainval_raw)
    rf_preds = rf_final.predict(X_test_rf)

    persistence_preds = []
    for state in s_test:
        ys = np.concatenate([y_train[s_train == state], y_val[s_val == state]])
        persistence_preds.append(ys[-1] if len(ys) > 0 else y_trainval_raw.mean())
    persistence_preds = np.array(persistence_preds)

    def evaluate(y_true, y_pred, name):
        return {
            'name': name,
            'R2': r2_score(y_true, y_pred),
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'MAPE': np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
        }

    test_loader = DataLoader(
        MaizeDataset(X_test_scaled, np.zeros(len(X_test_scaled)), s_test_oh),
        batch_size=Config.BATCH_SIZE,shuffle=False)

    results = [
        evaluate(y_test, persistence_preds, "🌱 Persistence Baseline"),
        evaluate(y_test, rf_preds, "🌲 Random Forest (tuned)"),
    ]

    dl_preds = {}
    for name, model in final_dl_models.items():
        preds = predict(model, test_loader, Config.DEVICE)
        dl_preds[name] = preds
        results.append(evaluate(y_test, preds, f"🧠 {name} (tuned)"))

    ensemble_preds = np.mean(list(dl_preds.values()), axis=0)
    results.append(evaluate(y_test, ensemble_preds, "🎯 Ensemble (DL avg)"))

    results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
    print("\n" + "=" * 70)
    print("🏆 Final Results — Version B (with lag features)")
    print("=" * 70)
    print(results_df.to_string(index=False))

    results_df.to_csv('results/metrics_versionB.csv', index=False)

    best = results_df.iloc[0]
    persistence_r2 = results_df[
        results_df['name'].str.contains('Persistence')
    ]['R2'].values[0]
    print(f"\n🥇 Best model: {best['name']}")
    print(f"📈 R²: {best['R2']:.4f}")
    print(f"📉 Improvement over persistence: {best['R2'] - persistence_r2:+.4f}")
    print("\n✅ Version B done. Results saved to results/metrics_versionB.csv")
    print("=" * 70)


if __name__ == "__main__":
    main()

🌽 Maize Yield Forecasting — Version B (with lag features)
🔥 Device: cpu

📊 Data after lags: (697, 25) | States: 37
🔧 Sequences: X=(512, 5, 8), y=(512,)
🧾 Features used: ['LST', 'NDVI', 'Precipitation', 'Temperature', 'year_sin', 'year_cos', 'value_lag1', 'value_lag2']
✂️  Train: 327 | Val: 74 | Test: 111
🏷️  State one-hot dim: 37

🔎 Tuning LSTM...
   trial 1/5: val_loss=0.0938 | params={'hidden_size': 128, 'num_layers': 1, 'dropout': 0.1, 'lr': 0.0005}
   trial 2/5: val_loss=0.1070 | params={'hidden_size': 128, 'num_layers': 1, 'dropout': 0.1, 'lr': 0.0005}
   trial 3/5: val_loss=0.1324 | params={'hidden_size': 32, 'num_layers': 2, 'dropout': 0.3, 'lr': 0.001}
   trial 4/5: val_loss=0.0966 | params={'hidden_size': 64, 'num_layers': 1, 'dropout': 0.1, 'lr': 0.0005}
   trial 5/5: val_loss=0.1194 | params={'hidden_size': 64, 'num_layers': 2, 'dropout': 0.2, 'lr': 0.0005}
   ✅ Best LSTM: val_loss=0.0938 | {'hidden_size': 128, 'num_layers': 1, 'dropout': 0.1, 'lr': 0.0005}

🔎 Tuning GRU...
